---
### 1. Imports

In [1]:
import json
import os
import sys
import uuid

import requests
from fastapi import HTTPException

In [2]:
sys.path.append(os.path.abspath(".."))
from wforum.db import api
from wforum.db.schemas import LoginRequest, SignupRequest

---
### 2. Create test account

In [3]:
signup_request = SignupRequest(
    username="test_user", email="test@user.me", password="test1234"
)
try:
    api.auth.signup(signup_request)
    print(f"User created:\n{signup_request.username}")
except HTTPException as e:
    assert e.status_code == 409
    print(f"Signup failed:\n{e.detail}")

Signup failed:
Username or email already exists


In [4]:
login_request = LoginRequest(username="test_user", password="test1234")

resp = api.auth.login(login_request)
token = resp.result["access_token"]  # type: ignore
print(f"Access token: <{len(token)} characters>")

Access token: <129 characters>


In [6]:
api.topics.create_topic_inner(
    topic_name="Valorant",
    description="Don't call you a Valorant player if you never forgot to check corner",
)

ResponseModel(status='OK', result=Topic(name='Valorant', description="Don't call you a Valorant player if you never forgot to check corner", stats=TopicStats()), message='new topic created successfully')

In [ ]:
api.topics.follow_topic("Valorant", "test_user")

'followed'

---
### 3. Request templates

In [8]:
BASE = "http://localhost:8080/mcp/"

#### 3.1. Headers templates

In [9]:
HEADERS_COMMON = {
    "Accept": "application/json, text/event-stream",
    "Content-Type": "application/json",
    "Mcp-Session-Id": str(uuid.uuid4()),
}

UNAUTHORIZED_HEADERS = {
    **HEADERS_COMMON,
    "Authorization": "Bearer random_token",
}

AUTHORIZED_HEADERS = {
    **HEADERS_COMMON,
    "Authorization": f"Bearer {token}",
}

#### 3.2. Payload templates

In [10]:
PAYLOAD_COMMON = {
    "jsonrpc": "2.0",
    "id": "1",
    "method": "tools/call",
}


def make_payload(tool_name: str, **arguments) -> dict:
    return {**PAYLOAD_COMMON, "params": {"name": tool_name, "arguments": arguments}}

---
### 4. `send_request` function

In [11]:
def send_request(headers: dict, payload: dict):
    resp = requests.post(BASE, headers=headers, json=payload, timeout=10)

    # check if response is ok
    if resp.ok:
        data = json.loads(resp.text.partition("data: ")[2])
        return "\n\n".join(t["text"] for t in data["result"]["content"])
    else:
        print("Request failed")
        print("Status code:", resp.status_code)
        return resp.text

---
### 5. Testing MCP Tools

In [14]:
# Test invalid request
text = send_request(
    AUTHORIZED_HEADERS,
    make_payload("create_post", title="Hello World", content="This is my first post!"),
)
print("Response text:")
print(text)

Response text:
{
  "topic": "Valorant",
  "username": "test_user"
}
